# Main script with experiments

In [6]:
import torch
import torch.nn as nn
import itertools

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import QuadraticMLP
from src.utils import build_sampler, data_load_and_prep, evaluate
from src.training import train

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
iris_train_loader, iris_test_loader = data_load_and_prep("iris", test_size=0.3, random_state=42, batch_size='full')
bc_train_loader, bc_test_loader = data_load_and_prep("breast_cancer", test_size=0.3, random_state=42, batch_size='full')
digits_train_loader, digits_test_loader = data_load_and_prep("digits", test_size=0.3, random_state=42, batch_size='full')

## Neural nets

### Iris

In [8]:
opt_param_grid = {
   
    'device': ['cpu'],
    'sampler': [build_sampler(mode="simulated")],
    'subset_size': [12, 24, 36],
    'step_size': [0.01, 0.05, 0.1],
    'num_reads': [50, 100, 200],
}

model_param_grid = {
    'input_dim': [4],
    'output_dim': [3],
    'c_device': ['cpu'],
    'hidden_dim': [[8], [16], [32], [16, 16], [32, 32], [64, 64], [16, 16, 16], [32, 32, 32], [64, 64, 64]],
    'activation': [nn.ReLU, nn.Tanh],
}

In [ ]:
def grid_search(train_loader: torch.utils.data.DataLoader, 
                test_loader: torch.utils.data.DataLoader, 
                optimizer_cls: torch.optim.Optimizer|QuadraticAnnealingOptimizer,
                experiment_name: str,
                opt_param_grid: dict, 
):
    """Function to perform grid search over hyperparameters."""
    loss_fn = nn.CrossEntropyLoss()

    keys, values = zip(*opt_param_grid.items())
    for combination in itertools.product(*values):
        params = dict(zip(keys, combination))

        model = QuadraticMLP(4, [32, 16], 3)
        optimizer = optimizer_cls(model=model, **params)

        train(model=model, 
              train_loader=train_loader, 
              test_loader=test_loader, 
              loss_fn=loss_fn,
              optimizer=optimizer, 
              epochs=50,
              experiment_name=experiment_name,
              c_device=params['device'],
              run_name=f"subset_{params['subset_size']}_step_{params['step_size']}_reads_{params['num_reads']}"
              )

In [11]:
grid_search(train_loader=iris_train_loader, 
            test_loader=iris_test_loader,
            optimizer_cls=QuadraticAnnealingOptimizer,
            experiment_name="sim-anneal-iris-hyper",
            opt_param_grid=opt_param_grid,
            )

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 000 | train_loss=1.0569 | train_acc=0.438 | test_loss=1.0463 | test_acc=0.467 | 
Epoch 005 | train_loss=0.9959 | train_acc=0.629 | test_loss=0.9869 | test_acc=0.667 | 
Epoch 010 | train_loss=0.9387 | train_acc=0.676 | test_loss=0.9320 | test_acc=0.689 | 
Epoch 015 | train_loss=0.8820 | train_acc=0.676 | test_loss=0.8774 | test_acc=0.667 | 
Epoch 020 | train_loss=0.8269 | train_acc=0.676 | test_loss=0.8240 | test_acc=0.667 | 
Epoch 025 | train_loss=0.7753 | train_acc=0.686 | test_loss=0.7739 | test_acc=0.667 | 
Epoch 030 | train_loss=0.7276 | train_acc=0.686 | test_loss=0.7280 | test_acc=0.667 | 
Epoch 035 | train_loss=0.6830 | train_acc=0.686 | test_loss=0.6862 | test_acc=0.689 | 
Epoch 040 | train_loss=0.6417 | train_acc=0.714 | test_loss=0.6473 | test_acc=0.711 | 


2026/03/12 19:22:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 19:22:00 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 19:22:00 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 045 | train_loss=0.6036 | train_acc=0.733 | test_loss=0.6119 | test_acc=0.711 | 
Epoch 049 | train_loss=0.5751 | train_acc=0.771 | test_loss=0.5857 | test_acc=0.733 | 
Epoch 000 | train_loss=1.0558 | train_acc=0.343 | test_loss=1.0528 | test_acc=0.333 | 
Epoch 005 | train_loss=0.9968 | train_acc=0.667 | test_loss=0.9984 | test_acc=0.667 | 
Epoch 010 | train_loss=0.9406 | train_acc=0.667 | test_loss=0.9472 | test_acc=0.644 | 
Epoch 015 | train_loss=0.8846 | train_acc=0.667 | test_loss=0.8966 | test_acc=0.644 | 
Epoch 020 | train_loss=0.8297 | train_acc=0.667 | test_loss=0.8472 | test_acc=0.667 | 
Epoch 025 | train_loss=0.7777 | train_acc=0.695 | test_loss=0.8005 | test_acc=0.667 | 
Epoch 030 | train_loss=0.7299 | train_acc=0.714 | test_loss=0.7577 | test_acc=0.667 | 
Epoch 035 | train_loss=0.6860 | train_acc=0.762 | test_loss=0.7182 | test_acc=0.689 | 
Epoch 040 | train_loss=0.6453 | train_acc=0.790 | test_loss=0.6813 | test_acc=0.733 | 


2026/03/12 19:22:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 19:22:04 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 19:22:04 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


...

Epoch 049 | train_loss=0.0000 | train_acc=1.000 | test_loss=0.7290 | test_acc=0.956 | 
Epoch 000 | train_loss=0.8233 | train_acc=0.895 | test_loss=0.8440 | test_acc=0.800 | 
Epoch 005 | train_loss=0.1991 | train_acc=0.952 | test_loss=0.2913 | test_acc=0.867 | 
Epoch 010 | train_loss=0.0571 | train_acc=0.981 | test_loss=0.1360 | test_acc=0.911 | 
Epoch 015 | train_loss=0.0131 | train_acc=1.000 | test_loss=0.2541 | test_acc=0.911 | 
Epoch 020 | train_loss=0.0026 | train_acc=1.000 | test_loss=0.3871 | test_acc=0.911 | 
Epoch 025 | train_loss=0.0008 | train_acc=1.000 | test_loss=0.4828 | test_acc=0.911 | 
Epoch 030 | train_loss=0.0002 | train_acc=1.000 | test_loss=0.5746 | test_acc=0.911 | 
Epoch 035 | train_loss=0.0001 | train_acc=1.000 | test_loss=0.6417 | test_acc=0.911 | 
Epoch 040 | train_loss=0.0000 | train_acc=1.000 | test_loss=0.6826 | test_acc=0.911 | 
Epoch 045 | train_loss=0.0000 | train_acc=1.000 | test_loss=0.8041 | test_acc=0.911 | 


2026/03/12 19:24:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 19:24:08 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 19:24:08 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 049 | train_loss=0.0000 | train_acc=1.000 | test_loss=0.9927 | test_acc=0.911 | 
Epoch 000 | train_loss=0.8033 | train_acc=0.819 | test_loss=0.8220 | test_acc=0.756 | 
Epoch 005 | train_loss=0.1898 | train_acc=0.971 | test_loss=0.2858 | test_acc=0.867 | 
Epoch 010 | train_loss=0.0544 | train_acc=0.981 | test_loss=0.1640 | test_acc=0.911 | 
Epoch 015 | train_loss=0.0191 | train_acc=1.000 | test_loss=0.2312 | test_acc=0.889 | 
Epoch 020 | train_loss=0.0045 | train_acc=1.000 | test_loss=0.3834 | test_acc=0.911 | 
Epoch 025 | train_loss=0.0013 | train_acc=1.000 | test_loss=0.5155 | test_acc=0.911 | 
Epoch 030 | train_loss=0.0003 | train_acc=1.000 | test_loss=0.6091 | test_acc=0.911 | 
Epoch 035 | train_loss=0.0001 | train_acc=1.000 | test_loss=0.7054 | test_acc=0.911 | 
Epoch 040 | train_loss=0.0000 | train_acc=1.000 | test_loss=0.8856 | test_acc=0.911 | 
Epoch 045 | train_loss=0.0000 | train_acc=1.000 | test_loss=0.9968 | test_acc=0.911 | 


2026/03/12 19:24:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 19:24:18 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 19:24:18 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 049 | train_loss=0.0000 | train_acc=1.000 | test_loss=1.1792 | test_acc=0.911 | 
